# cls_compare2.ipynb
**Does:** Experiment 2 - prior-shift correction, description features and stacked models, chosen by cross-validation, plus the λ trade-off (Part E).
**Uses:** `src/scale.py`, `src/piv_compare.py`, `src/cls_compare2.py` and the `cls_tables/` built by cls_compare.ipynb Part A.
**Outputs:** `cls2_outputs/<n>/` (base-model scores, cross-validation grid, validation and test results, figures).

## Settings

In [ ]:
# settings for this run (GPU with High-RAM needed for Part C)
DATA_NAME, RARE_NAME = 'run_13um', 'run_13um_rare'   # only needed if the tables have to be built
RUN_ID      = None               # None = new output folder; a number = continue that run
CLS_DATA    = 'cls_tables'       # tables built by cls_compare.ipynb
SOURCE, QF_FILE, QF_TAU, QF_TRAIN_STEP = 'pipeline', 'hybrid_segment_outputs/quality_filter.joblib', 0.3, 2
MOTION, TRACKER, EVERY_TRAIN, PIV_TRACKS = 'ransac', 'tracking', 3, None
SCHEME      = 'amaral7u'
FEATURE_MODELS = ['logreg', 'svm', 'lgbm', 'extratrees']
CNN_MODELS  = ['hybrid', 'resnet18']
CNN_EPOCHS  = {'hybrid': 60, 'resnet18': 30, 'mobilenet': 30, 'efficientnet': 30}
IMAGENET_SIZE = 128
CNN_MAX_PER_EPOCH = 40000
N_FOLDS     = 4                  # folds over source sprites
CLASS_POWER = 0.5
STACKERS    = ['stack_lr', 'stack_gbm']
SELECT_BY   = 'acc_bal_hmean'    # chosen by cross-validation on the training particles
MIN_ACCURACY = None
LAMBDAS     = (0.0, 0.25, 0.5, 0.75, 1.0)   # weight of the deployment prior (1 = accuracy, 0 = balanced)
GATES       = (False, True)      # size gate off / on
K_CHOICES   = (3, 10, None)      # frames per particle
PRIOR_MODES = ('natural',)
REJECT      = 0.5
CARBON_LAMBDA = 1.0
SIZE_MODEL  = False
CARBON_SCALES = 'train_oof'

## Setup

In [ ]:
# mount Drive, load the scripts, set the pixel scale and create the output folder
import os, sys, glob, json, shutil, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display
pd.set_option('display.max_columns', None, 'display.width', 250)

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/snocam_pipeline'
DATA, RARE = f'/content/{DATA_NAME}', f'/content/{RARE_NAME}'
sh = get_ipython().system
sh('pip install -q timm lightgbm xgboost openpyxl openpiv')
shutil.copytree(f'{DRIVE}/src', '/content/src', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
sys.path.append('/content/src')


def restore(name):
    # copy a dataset or table folder from Drive to /content (from its .tar if there is one)
    if not os.path.exists(f'/content/{name}'):
        if os.path.exists(f'{DRIVE}/{name}.tar'):
            sh(f'tar -xf "{DRIVE}/{name}.tar" -C /content')
        elif os.path.exists(f'{DRIVE}/{name}'):
            shutil.copytree(f'{DRIVE}/{name}', f'/content/{name}')
    return os.path.exists(f'/content/{name}')


import scale as SC
CLS = f'/content/{CLS_DATA}'
restore(CLS_DATA)
RES_FILE, DONE_FILE = f'{CLS}/resolution.json', f'{CLS}/tables_complete.json'
if os.path.exists(RES_FILE):            # tables already built: use their pixel scale
    r = json.load(open(RES_FILE))
    res = SC.configure(r['um_per_px'], min_area=r['min_area_px'], min_area_mode=r['min_area_mode'],
                       canvas=r['canvas_px'], source=RES_FILE)
else:                                   # otherwise read it from the data
    assert restore(DATA_NAME), f'no {DATA_NAME} in {DRIVE}: run pipeline.ipynb Part A first'
    res = SC.configure_from_data(DATA)
print('resolution:', res)
import torch
GPU = torch.cuda.is_available()

import cls_compare2 as cc
os.makedirs(f'{DRIVE}/cls2_outputs', exist_ok=True)
if RUN_ID is None:
    used = [int(n) for n in os.listdir(f'{DRIVE}/cls2_outputs') if n.isdigit()]
    RUN_ID = max(used) + 1 if used else 0
OUT = f'{DRIVE}/cls2_outputs/{RUN_ID}'
for sub in ('scores', 'models'):
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)
SETTINGS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, tuple, dict, type(None)))}
json.dump(dict(SETTINGS, resolution=res), open(f'{OUT}/settings.json', 'w'), indent=1, default=str)
print('outputs:', OUT, '| GPU:', GPU)


def backup_desc():
    # keep a copy of the cached description features on Drive
    os.makedirs(f'{DRIVE}/{CLS_DATA}_desc', exist_ok=True)
    for f in glob.glob(f'{CLS}/*_desc.npy'):
        shutil.copy(f, f'{DRIVE}/{CLS_DATA}_desc/')


def restore_desc():
    for f in glob.glob(f'{DRIVE}/{CLS_DATA}_desc/*_desc.npy'):
        if not os.path.exists(f'{CLS}/{os.path.basename(f)}'):
            shutil.copy(f, CLS)


def spath(name, part):
    return f'{OUT}/scores/{name}_{part}.npy'

## Status

In [ ]:
# what is already built and scored
if os.path.exists(CLS):
    restore_desc()
built = sorted(os.path.basename(f)[:-4] for f in glob.glob(f'{CLS}/*.pkl'))
print(f'particle tables in {CLS_DATA}:', pd.Series([b.split('_')[-1] for b in built]).value_counts().to_dict(),
      '(complete)' if os.path.exists(DONE_FILE) else '-> incomplete: run cls_compare Part A, or Part A here')
print('description features cached:', len(glob.glob(f'{CLS}/*_desc.npy')), '/', len(built))
print('base models done:', sorted({os.path.basename(f)[:-8] for f in glob.glob(f'{OUT}/scores/*_val.npy')}))
print('final configuration saved:', os.path.exists(f'{OUT}/final_config.json'))

## Helpers

In [ ]:
# helper functions used by every Part (run once per session)
COLS = ['accuracy', 'balanced_accuracy', 'acc_bal_hmean', 'flux_balanced_accuracy', 'macro_f1',
        'carbon_weighted_accuracy', 'accuracy_esd_ge_500um', 'carbon_error_found', 'carbon_error_all', 'n_particles']
_cache = {}

def tables(split, crops=False):
    key = (split, crops)
    if key not in _cache:
        T, C = cc.load_tables(CLS, (split,), crops=crops)
        T = cc.add_labels(cc.add_description(CLS, T), SCHEME)
        _cache[key] = (T, C)
    return _cache[key]

def truth_arr():
    if 'ta' not in _cache:
        _cache['ta'] = cc.truth_arrivals(CLS, ('val', 'test'))
    return _cache['ta']

def classes():
    if 'classes' not in _cache:
        _cache['classes'] = np.array(sorted(tables('train')[0].label.unique()))
    return _cache['classes']

def prior_nat():
    if 'prior' not in _cache:
        _cache['prior'] = cc.natural_prior(tables('train')[0], classes()).values
    return _cache['prior']

def folds():
    if 'folds' not in _cache:
        _cache['folds'] = cc.cv_folds(tables('train')[0], N_FOLDS)
    return _cache['folds']

def bases():
    return [n for n in FEATURE_MODELS + CNN_MODELS if os.path.exists(spath(n, 'val'))]

def train_parts():
    # training particles (one per truth particle; all its frames): out-of-fold base scores, size, truth
    key = ('train_parts', tuple(bases()))
    if key not in _cache:
        tr = tables('train')[0]
        idx, Zp, esd = cc.particle_inputs(tr, [np.load(spath(n, 'oof')) for n in bases()], tr.pkey.values)
        g = tr.groupby('pkey')
        info = pd.DataFrame({'y': g.label.first(), 'uid': g.uid.first(), 'source_uid': g.source_uid.first(),
                             'seq_id': g.seq_id.first(), 'carbon_true': g.carbon_true.first()}).reindex(idx)
        info['esd_um'] = esd
        info['group'] = np.where(info.uid > 0, info.source_uid.astype(str), 'noise:' + info.seq_id)
        pool = info.y.value_counts(normalize=True)
        info['w_acc'] = (pd.Series(prior_nat(), index=classes()) / pool).reindex(info.y).values   # natural mix
        _cache[key] = (idx, Zp, esd, info)
    return _cache[key]

def eval_parts(split, K):
    key = ('eval_parts', split, K, tuple(bases()))
    if key not in _cache:
        T = tables(split)[0]
        _cache[key] = cc.particle_inputs(T, [np.load(spath(n, split)) for n in bases()], cc.track_key(T), K)
    return _cache[key]

STK = {}                                   # fitted stackers (Part D)

def model_z(name, where, K=None):
    # particle scores of a model on the training particles out-of-fold ('cv') or on 'val' / 'test'
    if where == 'cv':
        idx, Zp, esd, info = train_parts()
        if name in cc.STACKERS:
            return np.load(spath(name, 'oof_particles'))
        return Zp[bases().index(name)] if name in bases() else np.mean(Zp, 0)
    idx, Zp, esd = eval_parts(where, K)
    if name in cc.STACKERS:
        return cc.stacker_scores(STK[name], cc.stack_inputs(Zp, esd), classes())
    return Zp[bases().index(name)] if name in bases() else np.mean(Zp, 0)

def cv_eval(name, lam, gate):
    idx, Zp, esd, info = train_parts()
    real = (info.uid > 0).values
    g = GATE if gate else None
    pen = cc.gate_penalty(classes(), esd, g) if g else None
    pred, _ = cc.decide_z(model_z(name, 'cv'), classes(), lam, np.log(prior_nat()), REJECT, pen)
    return cc.cv_metrics(pred[real], info.y.values[real], info.w_acc.values[real])

def evaluate(name, split, K, lam, gate, prior_mode='natural', scales=None, sm=None, carbon_lam=None):
    idx, Zp, esd = eval_parts(split, K)
    return cc.evaluate_z(tables(split)[0], model_z(name, split, K), idx, classes(), lam, prior_nat(), K, REJECT,
                         GATE if gate else None, prior_mode, sm=sm, scales=scales, truth_arr=truth_arr(),
                         scheme=SCHEME, carbon_lam=CARBON_LAMBDA if carbon_lam is None else carbon_lam,
                         carbon_gate=GATE)

def run_config(c, split, carbon=None):
    carbon = carbon or {}
    return evaluate(c['model'], split, c['K'], c['lam'], c['gate'], c['prior_mode'], carbon.get('scales'),
                    carbon.get('size_model'))

def show(df, sort=None):
    d = df.copy()
    pc = [c for c in d.columns if c in COLS + ['cv_accuracy', 'cv_balanced', 'cv_hmean'] and c != 'n_particles']
    d[pc] = (d[pc].astype(float) * 100).round(1)
    display(d.sort_values(sort, ascending=False) if sort else d)

GATE = json.load(open(f'{OUT}/final_config.json'))['gate_ranges'] if os.path.exists(f'{OUT}/final_config.json') else None
for kind in STACKERS:                      # after a restart: reload the fitted meta-models
    if os.path.exists(f'{OUT}/models/{kind}.joblib'):
        import joblib
        STK[kind] = joblib.load(f'{OUT}/models/{kind}.joblib')
print('helpers ready')

## Part A: particle tables and description features

In [ ]:
# restore (or build) the particle tables and measure the description features once per sequence
if not os.path.exists(DONE_FILE):
    restore(DATA_NAME); restore(RARE_NAME)
    import piv_compare as pc
    pc.use_hybrid_segment(f'{DRIVE}/{QF_FILE}', QF_TAU, data_dir=DATA, frame_step=QF_TRAIN_STEP)
    os.makedirs(CLS, exist_ok=True)
    json.dump(SC.summary(), open(RES_FILE, 'w'), indent=1)
    seqs = sorted(glob.glob(f'{DATA}/seq*_*')) + (sorted(glob.glob(f'{RARE}/rare*_train')) if RARE else [])
    tracks_dir = f'{DRIVE}/{PIV_TRACKS}' if PIV_TRACKS else None
    for d in seqs:
        sid = os.path.basename(d)
        if not os.path.exists(f'{CLS}/{sid}.pkl'):
            tcsv = f'{tracks_dir}/{sid}/{MOTION}__{TRACKER}.csv' if tracks_dir else None
            cc.build_sequence(d, CLS, SOURCE, EVERY_TRAIN if sid.endswith('_train') else 1,
                              track=not sid.endswith('_train'), tracks_csv=tcsv, motion=MOTION, tracker=TRACKER)
    json.dump(dict(sequences=[os.path.basename(d) for d in seqs], resolution=SC.summary()), open(DONE_FILE, 'w'), indent=1)
    sh(f'tar -cf "/content/{CLS_DATA}.tar" -C /content {CLS_DATA} && mv "/content/{CLS_DATA}.tar" "{DRIVE}/"')
# description features: a few ms per crop, cached as <seq>_desc.npy
for split in ('train', 'val', 'test'):
    t0 = time.time()
    T = tables(split)[0]
    print(split, len(T), 'detections', f'({time.time() - t0:.0f} s)')
backup_desc()

In [ ]:
# do the description features separate the classes? median per true class
tr = tables('train')[0]
p = tr[tr.uid > 0].groupby('pkey').agg({**{f: 'median' for f in cc.DESC_FEATURES + ['esd_um', 'aspect', 'solidity']}, 'label': 'first'})
display(p.groupby('label').median().T.round(2))
print('training particles per class:', p.label.value_counts().to_dict())

## Part B: feature models (out-of-fold)

In [ ]:
# train every feature model on N_FOLDS folds (out-of-fold scores) and on all training particles
tr, Tv, Tt = tables('train')[0], tables('val')[0], tables('test')[0]
cl = classes()
for name in FEATURE_MODELS:
    if os.path.exists(spath(name, 'val')):
        continue
    for k, (a, b) in enumerate(folds()):
        f = f'{OUT}/scores/{name}_fold{k}.npz'
        if os.path.exists(f):
            continue
        t0 = time.time()
        m = cc.fit_feature_model(name, tr.iloc[a], power=CLASS_POWER, seed=k)
        np.savez(f, oof=cc.feature_model_scores(m, tr.iloc[b], cl), val=cc.feature_model_scores(m, Tv, cl),
                 test=cc.feature_model_scores(m, Tt, cl))
        print(f'{name} fold {k}: {time.time() - t0:.0f} s', flush=True)
    oof = np.zeros((len(tr), len(cl)), np.float32)
    parts = [np.load(f'{OUT}/scores/{name}_fold{k}.npz') for k in range(N_FOLDS)]
    for (a, b), d in zip(folds(), parts):
        oof[b] = d['oof']
    np.save(spath(name, 'oof'), oof)
    np.save(spath(name, 'test'), np.mean([d['test'] for d in parts], 0))
    np.save(spath(name, 'val'), np.mean([d['val'] for d in parts], 0))   # written last: marks the model done
_cache.clear()
rows = {n: cv_eval(n, 0.5, False) for n in bases()}
show(pd.DataFrame(rows).T, 'acc_bal_hmean')

## Part C: CNNs (GPU)

In [ ]:
# train every CNN on N_FOLDS folds with weight averaging
assert GPU, 'Part C needs a GPU: Runtime -> Change runtime type -> T4 GPU'
trC, Ctr = tables('train', crops=True)
Tv, Cv = tables('val', crops=True)
Tt, Ct = tables('test', crops=True)
cl = classes()
print('crops in memory:', Ctr.shape, Cv.shape, Ct.shape)
for arch in CNN_MODELS:
    if os.path.exists(spath(arch, 'val')):
        continue
    for k, (a, b) in enumerate(folds()):
        f = f'{OUT}/scores/{arch}_fold{k}.npz'
        if os.path.exists(f):
            continue
        t0 = time.time()
        net = cc.train_cnn(arch, trC.iloc[a], Ctr[a], epochs=CNN_EPOCHS[arch], power=CLASS_POWER, seed=k,
                           max_per_epoch=CNN_MAX_PER_EPOCH, input_size=IMAGENET_SIZE, va=Tv, Cva=Cv,
                           log=lambda r: print(f'{arch} f{k}', r, flush=True) if 'val_balanced' in r else None)
        cc.save_cnn(net, f'{OUT}/models/{arch}_fold{k}.pt')
        net.history_.to_csv(f'{OUT}/models/{arch}_fold{k}_history.csv', index=False)
        np.savez(f, oof=cc.cnn_scores(net, trC.iloc[b], Ctr[b], cl), val=cc.cnn_scores(net, Tv, Cv, cl),
                 test=cc.cnn_scores(net, Tt, Ct, cl))
        print(f'{arch} fold {k}: {time.time() - t0:.0f} s', flush=True)
        del net; torch.cuda.empty_cache()
    oof = np.zeros((len(trC), len(cl)), np.float32)
    parts = [np.load(f'{OUT}/scores/{arch}_fold{k}.npz') for k in range(N_FOLDS)]
    for (a, b), d in zip(folds(), parts):
        oof[b] = d['oof']
    np.save(spath(arch, 'oof'), oof)
    np.save(spath(arch, 'test'), np.mean([d['test'] for d in parts], 0))
    np.save(spath(arch, 'val'), np.mean([d['val'] for d in parts], 0))
_cache.pop(('train_parts', tuple(bases())), None)
fig, ax = plt.subplots(figsize=(7, 4))
for f in sorted(glob.glob(f'{OUT}/models/*_history.csv')):
    h = pd.read_csv(f).dropna(subset=['val_balanced'])
    ax.plot(h.epoch, h.val_balanced, marker='o', label=os.path.basename(f)[:-12])
ax.set_xlabel('epoch'); ax.set_ylabel('val balanced accuracy (logged only)'); ax.legend(fontsize=7); plt.show()

## Part D: stack and decision (cross-validation)

In [ ]:
# fit the stackers and choose model, lambda, size gate and frames per particle by cross-validation
import joblib
idx, Zp, esd, info = train_parts()
cl = classes()
real = (info.uid > 0).values
GATE = cc.size_gate(info.y.values[real], esd[real], cl)
print('size gate (um):', {k: (round(v[0]), round(v[1])) for k, v in GATE.items()})
X = cc.stack_inputs(Zp, esd)
for kind in STACKERS:
    if not os.path.exists(spath(kind, 'oof_particles')):
        np.save(spath(kind, 'oof_particles'), cc.stacker_oof(kind, X, info.y.values, info.group.values, cl))
        joblib.dump(cc.fit_stacker(kind, X, info.y.values, cl), f'{OUT}/models/{kind}.joblib')
    STK[kind] = joblib.load(f'{OUT}/models/{kind}.joblib')
MODELS = bases() + ['average'] + list(STACKERS)
grid = pd.DataFrame([dict(model=n, lam=l, gate=g, **{f'cv_{k}': v for k, v in cv_eval(n, l, g).items()})
                     for n in MODELS for l in LAMBDAS for g in GATES])
grid = grid.rename(columns={'cv_balanced_accuracy': 'cv_balanced', 'cv_acc_bal_hmean': 'cv_hmean'})
grid.to_csv(f'{OUT}/cv_grid.csv', index=False)
crit = {'acc_bal_hmean': 'cv_hmean', 'accuracy': 'cv_accuracy', 'balanced_accuracy': 'cv_balanced'}[SELECT_BY]
ok = grid if MIN_ACCURACY is None else grid[grid.cv_accuracy >= MIN_ACCURACY]
if MIN_ACCURACY is not None:
    crit = 'cv_balanced'
best = ok.loc[ok.groupby('model')[crit].idxmax()].set_index('model').sort_values(crit, ascending=False)
show(best)
win = best.index[0]
# validation: frames per particle and prior mode for the winner (and, for reference, every model's validation score)
vrows = []
for K in K_CHOICES:
    for pm in PRIOR_MODES:
        r, *_ = evaluate(win, 'val', K, best.loc[win, 'lam'], bool(best.loc[win, 'gate']), pm)
        vrows.append(dict(K=K, prior_mode=pm, **r))
V = pd.DataFrame(vrows)
show(V[['K', 'prior_mode'] + COLS])
vb = V.loc[V[SELECT_BY if MIN_ACCURACY is None else 'balanced_accuracy'].idxmax()]
final = dict(model=win, lam=float(best.loc[win, 'lam']), gate=bool(best.loc[win, 'gate']),
             K=None if pd.isna(vb.K) else int(vb.K), prior_mode=vb.prior_mode, select_by=SELECT_BY,
             min_accuracy=MIN_ACCURACY, scheme=SCHEME, reject=REJECT, carbon_lambda=CARBON_LAMBDA,
             gate_ranges=GATE, cv=best.loc[win, ['cv_accuracy', 'cv_balanced', 'cv_hmean']].astype(float).to_dict())
json.dump(final, open(f'{OUT}/final_config.json', 'w'), indent=1)
print(json.dumps({k: v for k, v in final.items() if k != 'gate_ranges'}, indent=1))

In [ ]:
# every model at its cross-validated setting, on validation
final = json.load(open(f'{OUT}/final_config.json'))
best = pd.read_csv(f'{OUT}/cv_grid.csv')
crit = 'cv_balanced' if final['min_accuracy'] is not None else {'acc_bal_hmean': 'cv_hmean', 'accuracy': 'cv_accuracy', 'balanced_accuracy': 'cv_balanced'}[final['select_by']]
if final['min_accuracy'] is not None:
    best = best[best.cv_accuracy >= final['min_accuracy']]
best = best.loc[best.groupby('model')[crit].idxmax()].set_index('model')
rows = {}
for n, b in best.iterrows():
    r, *_ = evaluate(n, 'val', final['K'], b.lam, bool(b.gate), final['prior_mode'])
    rows[n] = {**r, 'lam': b.lam, 'gate': b.gate, 'cv_hmean': b.cv_hmean}
VAL = pd.DataFrame(rows).T
VAL.to_csv(f'{OUT}/validation.csv')
show(VAL[COLS + ['lam', 'gate', 'cv_hmean']], 'acc_bal_hmean')

## Part E: checks

In [ ]:
# is cross-validation honest? CV vs validation scores
VAL = pd.read_csv(f'{OUT}/validation.csv', index_col=0)
grid = pd.read_csv(f'{OUT}/cv_grid.csv')
chk = pd.DataFrame({'cv_accuracy': [grid[(grid.model == n) & (grid.lam == VAL.loc[n, 'lam']) & (grid.gate == VAL.loc[n, 'gate'])].cv_accuracy.iloc[0] for n in VAL.index],
                    'val_accuracy': VAL.accuracy.values,
                    'cv_balanced': [grid[(grid.model == n) & (grid.lam == VAL.loc[n, 'lam']) & (grid.gate == VAL.loc[n, 'gate'])].cv_balanced.iloc[0] for n in VAL.index],
                    'val_balanced': VAL.balanced_accuracy.values}, index=VAL.index)
display((chk * 100).round(1).sort_values('val_balanced', ascending=False))
print('rank correlation CV vs val (balanced):', round(chk.cv_balanced.rank().corr(chk.val_balanced.rank()), 2))

In [ ]:
# accuracy / balanced-accuracy trade-off as lambda goes from 0 to 1
final = json.load(open(f'{OUT}/final_config.json'))
lams = np.linspace(0, 1.2, 13)
tr_c, va_c = [], []
for l in lams:
    tr_c.append(cv_eval(final['model'], l, final['gate']))
    va_c.append(evaluate(final['model'], 'val', final['K'], l, final['gate'], final['prior_mode'])[0])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, cur, t in ((ax[0], tr_c, 'cross-validation'), (ax[1], va_c, 'validation')):
    a.plot(lams, [c['accuracy'] for c in cur], 'o-', label='accuracy')
    a.plot(lams, [c['balanced_accuracy'] for c in cur], 's-', label='balanced accuracy')
    a.axhline(0.9, ls=':', c='k'); a.axhline(0.8, ls=':', c='grey'); a.axvline(final['lam'], c='r', alpha=.4)
    a.set_xlabel('lambda (weight of the deployment prior)'); a.set_title(t); a.legend(); a.set_ylim(0, 1)
plt.show()
display((pd.DataFrame(va_c, index=lams.round(2))[['accuracy', 'balanced_accuracy', 'acc_bal_hmean', 'carbon_weighted_accuracy']] * 100).round(1))

In [ ]:
# the chosen configuration per class and per size (validation)
final = json.load(open(f'{OUT}/final_config.json'))
r, info, pred, cp, per = run_config(final, 'val')
ta = info.true_arrival.values
d = pd.DataFrame({'true': cc.scheme_label(info.source_class.values, SCHEME)[ta], 'pred': pred.values[ta],
                  'esd': info.esd_um.values[ta], 'sprite': info.source_uid.values[ta]})
d['ok'] = d.true == d.pred
display(d.groupby('true').agg(n=('ok', 'size'), sprites=('sprite', 'nunique'), recall=('ok', 'mean')).round(3))
d['size'] = pd.cut(d.esd, [0, 200, 300, 500, 800, 1e6], labels=['<200', '200-300', '300-500', '500-800', '>800 um'])
display(d.groupby('size', observed=True).agg(n=('ok', 'size'), accuracy=('ok', 'mean')).round(3))
display(cc.confusion(info, pred.values, SCHEME))

In [ ]:
# how much the stack relies on each base model
if 'stack_lr' in STK:
    lr = STK['stack_lr'][-1]
    nb, nc = len(bases()), len(classes())
    W = np.abs(lr.coef_[:, :nb * nc]).reshape(len(lr.classes_), nb, nc).mean((0, 2))
    display(pd.Series(W, index=bases()).sort_values(ascending=False).round(3).to_frame('mean |weight|'))

## Part F: carbon

In [ ]:
# class used for carbon, carbon re-fit and carbon errors
final = json.load(open(f'{OUT}/final_config.json'))
idx, Zp, esd_p, info_p = train_parts()
real = (info_p.uid > 0).values
pen = cc.gate_penalty(classes(), esd_p, final['gate_ranges'])          # carbon always uses the size gate
cpred_tr, _ = cc.decide_z(model_z(final['model'], 'cv'), classes(), CARBON_LAMBDA, np.log(prior_nat()), REJECT, pen)
sm = cc.fit_size_model(tables('train')[0]) if SIZE_MODEL else None
if CARBON_SCALES == 'train_oof':
    scales = cc.fit_class_scales(cpred_tr[real], esd_p[real], info_p.carbon_true.values[real])
elif CARBON_SCALES == 'val':
    r, info, pred, cp, per = run_config(final, 'val')
    scales = cc.fit_class_scales(info.carbon_class.values[info.true_arrival.values], info.esd_um.values[info.true_arrival.values],
                                 info.carbon_true.values[info.true_arrival.values])
else:
    scales = {}
final['carbon'] = dict(scales=scales, size_model=sm, scales_from=CARBON_SCALES)
json.dump(final, open(f'{OUT}/final_config.json', 'w'), indent=1)
print('class carbon scales:', {k: round(v, 3) for k, v in scales.items()})
strat = {}
for name, carb in (('no scales', {'scales': {}, 'size_model': sm}), (f'scales from {CARBON_SCALES}', final['carbon'])):
    r, info, pred, cp, per = run_config(final, 'val', carb)
    strat[name] = {k: r[k] for k in ('carbon_error_found', 'carbon_error_all')}
display((pd.DataFrame(strat).T * 100).round(1))
r, info, pred, cp, per = run_config(final, 'val', final['carbon'])
ta = info.true_arrival.values
by = pd.DataFrame({'true': pd.Series(info.carbon_true.values[ta]).groupby(cc.scheme_label(info.source_class.values, SCHEME)[ta]).sum(),
                   'predicted': pd.Series(cp[ta]).groupby(info.carbon_class.values[ta]).sum()}).fillna(0) * 1000
display(by.round(3))
budget = cc.carbon_budget(info, cp, truth_arr())
budget.to_csv(f'{OUT}/carbon_budget_val.csv')
display(budget.round(2))

## Part G: final test (scored once)

In [ ]:
# final test: the chosen configuration on the test sequences, scored once
final = json.load(open(f'{OUT}/final_config.json'))
assert 'carbon' in final, 'run Part F first'
grid = pd.read_csv(f'{OUT}/cv_grid.csv')
ref = grid[grid.model == 'average'].sort_values('cv_hmean').iloc[-1]
configs = {f"chosen: {final['model']}": final,
           'reference: average': dict(final, model='average', lam=float(ref.lam), gate=bool(ref.gate))}
res, blocks = {}, {}
for lab, c in configs.items():
    r, info_t, pred_t, cp_t, per_t = run_config(c, 'test', final['carbon'])
    ci = cc.bootstrap_ci(info_t, pred_t.values, cp_t, truth_arr(), SCHEME)
    res[lab] = {**r, **{f'{k} 95% CI': f'{100 * lo:.1f} to {100 * hi:.1f}' for k, (lo, hi) in ci.items()}}
    blocks[lab] = (info_t, pred_t, cp_t, per_t)
TEST = pd.DataFrame(res).T
show(TEST)
for lab in TEST.index:
    print(f"{lab}: accuracy {100 * TEST.loc[lab, 'accuracy']:.1f} % (target 90), balanced {100 * TEST.loc[lab, 'balanced_accuracy']:.1f} % (target 80)")

In [ ]:
# test results per sequence, per class and the confusion matrix
lab = [l for l in blocks if l.startswith('chosen')][0]
info_t, pred_t, cp_t, per_t = blocks[lab]
seq_tab = cc.per_sequence(info_t, pred_t.values, cp_t, truth_arr(), SCHEME).set_index('seq_id')
conf = cc.confusion(info_t, pred_t.values, SCHEME)
budget = cc.carbon_budget(info_t, cp_t, truth_arr())
with pd.ExcelWriter(f'{OUT}/TEST_results.xlsx') as xw:
    TEST.to_excel(xw, sheet_name='final (scored once)')
    seq_tab.to_excel(xw, sheet_name='per test sequence')
    conf.to_excel(xw, sheet_name='confusion')
    per_t.to_frame('recall').to_excel(xw, sheet_name='recall by class')
    budget.to_excel(xw, sheet_name='carbon budget')
    pd.read_csv(f'{OUT}/validation.csv', index_col=0).to_excel(xw, sheet_name='validation, all models')
    pd.read_csv(f'{OUT}/cv_grid.csv').to_excel(xw, sheet_name='cv grid', index=False)
show(seq_tab[[c for c in COLS if c in seq_tab.columns]])
display(conf)
display(budget.round(2))
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
m = conf.div(conf.sum(1), axis=0)
ax[0].imshow(m.values, cmap='Blues', vmin=0, vmax=1)
ax[0].set_xticks(range(m.shape[1]), m.columns, rotation=60, ha='right'); ax[0].set_yticks(range(m.shape[0]), m.index)
ax[0].set_title('test confusion (row-normalised)')
per_t.sort_values().plot.barh(ax=ax[1], color='#2a78d6'); ax[1].set_xlim(0, 1); ax[1].set_title('test recall by class')
ta = info_t.true_arrival.values
cb = pd.DataFrame({'true': pd.Series(info_t.carbon_true.values[ta]).groupby(cc.scheme_label(info_t.source_class.values, SCHEME)[ta]).sum(),
                   'predicted': pd.Series(cp_t[ta]).groupby(info_t.carbon_class.values[ta]).sum()}).fillna(0) * 1000
cb.plot.bar(ax=ax[2], color=['#8a8a85', '#2a78d6']); ax[2].set_ylabel('carbon (ug)'); ax[2].set_title('test carbon by class (found particles)')
plt.tight_layout(); plt.savefig(f'{OUT}/test_summary.png', dpi=150); plt.show()
print('saved', f'{OUT}/TEST_results.xlsx')